# Imports

In [ ]:
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import seaborn as sns

## All Teams

In [ ]:
DATA_DIR = '/Users/chrischoi/Desktop/springboard/nhl-shots/data/raw'
os.listdir(DATA_DIR)

In [ ]:
# Load to df
all_teams = os.path.join(DATA_DIR, 'all_teams.csv')
mp_all = pd.read_csv(all_teams)

In [ ]:
# First looks at data
mp_all.info()
mp_all.describe(include='all')
mp_all.head()

In [ ]:
# Clean column names
def clean_column_names(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.replace("/", "_")
    )
    return df

mp_all = clean_column_names(mp_all)
mp_all.columns

In [ ]:
# Create date column in dt
mp_all['date'] = pd.to_datetime(mp_all['gamedate'].astype(str), format='%Y%m%d')

In [ ]:
# Create game key
mp_all['game_key'] = np.where(
    mp_all['home_or_away'] == 'HOME',
    mp_all['date'].dt.strftime('%Y%m%d') + '_' + mp_all['team'] + '_' + mp_all['opposingteam'],
    mp_all['date'].dt.strftime('%Y%m%d') + '_' + mp_all['opposingteam'] + '_' + mp_all['team']
)

In [ ]:
# Organize into dtypes
categorical_cols = mp_all.select_dtypes(include=['object']).columns.tolist()
numeric_cols = mp_all.select_dtypes(include=['number']).columns.tolist()
datetime_cols = mp_all.select_dtypes(include=['datetime']).columns.tolist()

categorical_cols, numeric_cols, datetime_cols

In [ ]:
# Check for missing values
missing_values = mp_all.isna().sum().sort_values(ascending=False)
print(missing_values[missing_values > 0])

In [ ]:
# playerteam and team are the same
mp_all['team'] = mp_all['team'].str.strip().str.upper()
mp_all['playerteam'] = mp_all['playerteam'].str.strip().str.upper()
print(mp_all[mp_all['playerteam'] != mp_all['team']])
mp_all.info()

# Master table

In [ ]:
master_cols = ['gameid', 'date', 'game_key']
mp_master = (
    mp_all[master_cols]
    .drop_duplicates('gameid')
)
mp_master = mp_master.reset_index(drop=True)
mp_master.head(), mp_master.nunique()

# Shots

In [ ]:
# Load shots to dfs
shots_to23 = os.path.join(DATA_DIR, 'shots_2015-2023.csv')
mp_shot_1 = pd.read_csv(shots_to23)
shots_24 = os.path.join(DATA_DIR, 'shots_2024.csv')
mp_shot_2 = pd.read_csv(shots_24)

shared_cols = list(set(mp_shot_1.columns) & set(mp_shot_2.columns))
mp_shot_1t = mp_shot_1[shared_cols]
mp_shot_2t = mp_shot_2[shared_cols]


mp_shots = pd.concat([mp_shot_1t, mp_shot_2t], ignore_index=True)

In [ ]:
# Shots first look
mp_shots.shape
mp_shots.info()
mp_shots.describe(include = 'all')

In [ ]:
# View all column names
mp_shots = clean_column_names(mp_shots)
print("\n".join(sorted(mp_shots.columns)))

In [ ]:
# Check for missing values
missing_values = mp_shots.isna().sum().sort_values(ascending=False)
print(missing_values[missing_values > 0])

In [ ]:
# Missing goalies
# print(mp_shots[mp_shots['goalieidforshot'].isna()])
filtered_goalies = mp_shots[mp_shots['goalieidforshot'].isna() & mp_shots['goalienameforshot'].isna()]
filtered_goalies[['goalieidforshot', 'goalienameforshot', 'game_id', 'season', 'hometeamcode', 'awayteamcode', 'teamcode', 'shootername']]

In [ ]:
# Carolina played emergency backup Dave Ayres, dropping these rows
mp_shots = mp_shots.drop(index=filtered_goalies.index)

In [ ]:
mp_shots[mp_shots['goalieidforshot'].isna() & mp_shots['goalienameforshot'].isna()]

In [ ]:
# Check for players/goalies who have more than one name assocated with each ID

bad_ids = mp_shots.groupby('goalieidforshot')['goalienameforshot'].nunique() != 1
bad_ids[bad_ids]
bad_pids = mp_shots.groupby('shooterplayerid')['shootername'].nunique() != 1
bad_pids[bad_pids]

In [ ]:
# Print goalie IDs w/ more than one name and the associated names
bad_id_list = bad_ids.index[bad_ids].tolist()
for id in bad_id_list:
    print(id)
    print(mp_shots.loc[mp_shots['goalieidforshot'] == id, 'goalienameforshot'].unique())

In [ ]:
# Fix using dictionary and map
gid_fix = {
    8475234.0: 'J.F. Berube',
    8477361.0: 'Calvin Petersen',
    8478965.0: 'Kenneth Appleby',
    8480022.0: 'Michael DiPietro'
}
mp_shots.loc[
    mp_shots['goalieidforshot'].isin(gid_fix.keys()),
    'goalienameforshot'
] = mp_shots['goalieidforshot'].map(gid_fix)

bad_gid_check = (mp_shots.groupby('goalieidforshot')['goalienameforshot'].nunique() != 1)
bad_gid_check[bad_gid_check]

In [ ]:
# Check for player IDs w/ more than one name and the associated names
bad_pid_list = bad_pids.index[bad_pids].tolist()
for id in bad_pid_list:
    print(id)
    print(mp_shots.loc[mp_shots['shooterplayerid'] == id, 'shootername'].unique())
    
name_counts = (
    mp_shots
    .groupby(['shooterplayerid', 'shootername'])
    .size()
    .reset_index(name='count')
)

# Create a list that saves only the name that appears more often for each player ID
standardized_names = (
    name_counts
    .sort_values(['shooterplayerid', 'count'], ascending=[True, False])
    .drop_duplicates('shooterplayerid')
)

# Zip dict and map the fix
pid_fix = dict(
    zip(standardized_names['shooterplayerid'], standardized_names['shooterplayerid'])
)

mp_shots['shooterplayerid'] = mp_shots['shooterplayerid'].map(pid_fix)


In [ ]:
missing_values = mp_shots.isna().sum().sort_values(ascending=False)
print(missing_values[missing_values > 0], (mp_shots['goalieidforshot'] == 0).sum())
print("\nThe number of goalie IDs that are equal to 0 and the number of missing goalie names now match.")


In [ ]:
# 6906 of the shots missing goalie names and IDs are shots on empty nets. 
empty_nets = mp_shots[
    (mp_shots['goalieidforshot'] == 0) &
    (mp_shots['goalienameforshot'].isna()) &
    (mp_shots['shotonemptynet'] == 1)
]
empty_nets = empty_nets.copy()
empty_nets['goalienameforshot'] = empty_nets['goalienameforshot'].fillna('Empty Net')

In [ ]:
# Drop missing values
mp_shots = mp_shots[~mp_shots['goalienameforshot'].isna()]           # Goalie IDs
mp_shots = pd.concat([mp_shots, empty_nets], ignore_index=True)      # Add back in empty net shots
mp_shots = mp_shots[~mp_shots['shooterleftright'].isna()]            # LH or RH shooter
mp_shots = mp_shots[~mp_shots['shottype'].isna()]                    # Shot type
mp_shots = mp_shots[~mp_shots['playerpositionthatdidevent'].isna()]  # Player Position
mp_shots.isna().sum().sort_values(ascending=False)

In [ ]:
# Export to csv to view all columns at once
mp_shots.sample(20, random_state=12).to_csv("20_shots", index=False)


In [ ]:
# Create gameid column in mp_shots, matching format of gameid found in mp_master
mp_shots['gameid'] = mp_shots['season'].astype(str) + mp_shots['isplayoffgame'].astype(int).astype(str) + mp_shots['game_id'].astype(str)
mp_shots['gameid'] = mp_shots['gameid'].astype(int)
# Check for consistentcy
print(mp_shots['gameid'].dtype)
print(mp_master['gameid'].dtype)

In [ ]:
# Using gameid column, merge date and game_key to mp_shots
mp_shots = mp_shots.merge(
    mp_master[['gameid', 'date', 'game_key']],
    on='gameid',
    how='left'
)


In [ ]:
mp_shots[['gameid', 'date', 'game_key']].isna().sum()

In [ ]:
# Saw a lot of NA values, identifying why
mp_shots.loc[mp_shots['isplayoffgame'] == 0, ['game_id', 'season', 'gameid', 'game_key', 'date']]

In [ ]:
blank_key = mp_shots[mp_shots['game_key'].isna()]
missing_values_regseason = mp_shots.loc[mp_shots['isplayoffgame'] == 0].isna().sum().sort_values(ascending = False)

In [ ]:
print(missing_values_regseason[0:10])
print(100 - (len(blank_key)/len(mp_shots) * 100))

In [ ]:
# Playoff games are the culprit; if I drop all shots taken during playoff games, I am left with 92.8% of shots taken,
# more than enough to build a strong model

In [ ]:
# Drop playoff games
mp_shots = mp_shots[mp_shots['isplayoffgame'] == 0]

In [ ]:
# Downcast
int_shots = mp_shots.select_dtypes('int64').columns
float_shots = mp_shots.select_dtypes('float64').columns

mp_shots[int_shots] = mp_shots[int_shots].apply(pd.to_numeric, downcast='integer')
mp_shots[float_shots] = mp_shots[float_shots].apply(pd.to_numeric, downcast='float')

obj_shots = mp_shots.select_dtypes(include=['object']).columns

for col in obj_shots:
    num_unique = mp_shots[col].nunique()
    num_total = len(mp_shots[col])
    
    if num_unique / num_total < 0.5:
        mp_shots[col] = mp_shots[col].astype('category')

In [ ]:
# View obj columns
mp_shots[obj_shots]# View columns with 2 or less unique variables
bool_shots = [col for col in mp_shots.columns if mp_shots[col].nunique() <=2]
mp_shots[bool_shots].head()

In [ ]:
# View columns with 2 or less unique variables
bool_shots = [col for col in mp_shots.columns if mp_shots[col].nunique() <=2]
mp_shots[bool_shots].head()

In [ ]:
# Convert team and shooterleftright, renaming columns for clarity
mp_shots['ishometeam'] = mp_shots['team'] == 'HOME'
####      mp_shots['shooterright'] = mp_shots['shooterleftright']
mp_shots = mp_shots.drop(columns = ['team', 'shooterleftright'])

# ID columns again with new names
bool_shots = [col for col in mp_shots.columns if mp_shots[col].nunique() <=2]

# Convert these columns to int8 -- previously used boolean but this was less compatible when doing analysis
mp_shots[bool_shots] = mp_shots[bool_shots].astype('int8')

In [ ]:
# Defragmentation and info
mp_shots.info()

mp_shots.to_csv("/Users/chrischoi/Desktop/springboard/nhl-shots/data/clean/mp_shots_clean.csv", index=False)
